# LFW Grad-CAM — 03. Saliency feature and faithfulness validation

사례를 고르기 전에 전체 표본 coverage, heatmap 유효성, 공간 특징과
occlusion control을 검증합니다. semantic landmark mask가 없으면 눈·볼·턱
수치를 추정하지 않고 결측으로 유지합니다.


In [4]:
# cell 1 : 실행 코드
from __future__ import annotations

from pathlib import Path
import sys

import yaml

PROJECT_ROOT = Path.cwd().resolve()
for candidate in (PROJECT_ROOT, *PROJECT_ROOT.parents):
    if (candidate / "research").is_dir() and (candidate / "configs").is_dir():
        PROJECT_ROOT = candidate
        break
else:
    raise RuntimeError("프로젝트 루트(C:/ronbun)를 찾을 수 없습니다.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

CONFIG_PATH = PROJECT_ROOT / "configs/experiments/step2_pytorch_gradcam.yaml"
CONFIG = yaml.safe_load(CONFIG_PATH.read_text(encoding="utf-8"))

MODEL_PROFILE = "arcface_ms1mv3_r100"     # arcface, adaface, magface 중 이번 실행 profile
MODE = "dev"               # 빠른 검증은 dev, 전체 논문 실행만 real
DATA_FRACTION = 1       # identity 단위 사용 비율; 0 < 값 <= 1
SEED = 42                  # 부분집합과 random control의 재현 seed
EXECUTE_STAGE = True      # 필수 입력을 채우고 이 단계 계산 시에만 True
WRITE_OUTPUTS = True      # 새 immutable artifact 저장 시에만 True

all_profiles = CONFIG["models"]["selected_profiles"] + CONFIG["models"].get("bridge_profiles", [])
available_profiles = CONFIG["models"]["profiles"]
blocked_profiles = CONFIG["models"].get("blocked_profiles", [])

if MODEL_PROFILE in blocked_profiles:
    raise RuntimeError(f"차단된 profile입니다: {MODEL_PROFILE}")
if MODEL_PROFILE not in available_profiles:
    raise ValueError(f"지원하지 않는 MODEL_PROFILE: {MODEL_PROFILE}")
PROFILE = available_profiles[MODEL_PROFILE]
MODEL_FAMILY = PROFILE["family"]
if MODE not in {"dev", "real"}:
    raise ValueError("MODE는 'dev' 또는 'real'이어야 합니다.")
if not 0.0 < DATA_FRACTION <= 1.0:
    raise ValueError("DATA_FRACTION은 (0, 1] 범위여야 합니다.")
if MODE == "real" and DATA_FRACTION != 1.0:
    raise ValueError("real 모드는 DATA_FRACTION=1.0이어야 합니다.")
if WRITE_OUTPUTS and not EXECUTE_STAGE:
    raise ValueError("WRITE_OUTPUTS=True이면 EXECUTE_STAGE도 True여야 합니다.")


In [5]:
# cell 2 : 실행 코드
import json
from pathlib import Path

import numpy as np
import pandas as pd

from research.explainability.gradcam import (
    read_population_saliency_features,
)

SALIENCY_ARTIFACT_DIR = None
VALIDATION_SUMMARY_OUTPUT_PATH = None

# 자동 기본값 및 경로 탐색
if SALIENCY_ARTIFACT_DIR is None or VALIDATION_SUMMARY_OUTPUT_PATH is None:
    lfw_runs_root = PROJECT_ROOT / CONFIG["run"]["root"] / "lfw"
    latest_saliency_manifests = sorted(lfw_runs_root.rglob("saliency_population/manifest.json"), key=lambda p: p.stat().st_mtime, reverse=True)
    if latest_saliency_manifests:
        sal_dir = latest_saliency_manifests[0].parent
        if SALIENCY_ARTIFACT_DIR is None:
            SALIENCY_ARTIFACT_DIR = sal_dir
        if VALIDATION_SUMMARY_OUTPUT_PATH is None:
            VALIDATION_SUMMARY_OUTPUT_PATH = sal_dir / "validation_summary.json"


In [6]:
# cell 3 : 실행 코드
if EXECUTE_STAGE:
    if SALIENCY_ARTIFACT_DIR is None:
        raise RuntimeError("SALIENCY_ARTIFACT_DIR를 지정하세요.")
    features = read_population_saliency_features(
        SALIENCY_ARTIFACT_DIR
    )
    status_counts = (
        features.groupby(
            [
                "saliency_target_eligible",
                "saliency_target_status",
                "heatmap_available",
            ],
            dropna=False,
        )
        .size()
        .rename("sample_count")
        .reset_index()
    )
    eligible = features["saliency_target_eligible"].astype(bool)
    if not features.loc[eligible, "heatmap_available"].astype(bool).all():
        raise RuntimeError("LOO 적격 표본 중 heatmap 누락이 있습니다.")
    if features.loc[~eligible, "heatmap_available"].astype(bool).any():
        raise RuntimeError("부적격 표본에 다른 target heatmap이 섞였습니다.")

    valid = eligible & features["gradcam_valid_heatmap"].fillna(False)
    validation_summary = {
        "sample_count": int(len(features)),
        "eligible_count": int(eligible.sum()),
        "eligible_fraction": float(eligible.mean()),
        "valid_heatmap_count": int(valid.sum()),
        "semantic_masked_sample_count": int(
            (
                features["semantic_region_mask_count"].fillna(0) > 0
            ).sum()
        ),
        "faithfulness_rows": int(
            features.get(
                "high_saliency_occlusion_score_drop",
                pd.Series(np.nan, index=features.index),
            ).notna().sum()
        ),
        "median_high_saliency_drop": float(
            features.loc[
                valid,
                "high_saliency_occlusion_score_drop",
            ].median()
        ),
        "median_low_saliency_drop": float(
            features.loc[
                valid,
                "low_saliency_occlusion_score_drop",
            ].median()
        ),
        "median_random_drop": float(
            features.loc[
                valid,
                "random_occlusion_score_drop",
            ].median()
        ),
    }
    if WRITE_OUTPUTS:
        if VALIDATION_SUMMARY_OUTPUT_PATH is None:
            raise RuntimeError("VALIDATION_SUMMARY_OUTPUT_PATH를 지정하세요.")
        destination = Path(VALIDATION_SUMMARY_OUTPUT_PATH).resolve()
        if destination.exists():
            raise FileExistsError(f"기존 결과를 덮어쓸 수 없습니다: {destination}")
        destination.parent.mkdir(parents=True, exist_ok=True)
        destination.write_text(
            json.dumps(
                validation_summary,
                ensure_ascii=False,
                indent=2,
            )
            + "\n",
            encoding="utf-8",
        )
else:
    status_counts = pd.DataFrame()
    validation_summary = {
        "status": "not_executed",
        "reason": "EXECUTE_STAGE=False",
    }
display(status_counts)
validation_summary


FileExistsError: 기존 결과를 덮어쓸 수 없습니다: C:\ronbun\runs\step2\lfw\arcface-7972a704552df378345f\population-c1b696495d2787e584b4639d\saliency_population\validation_summary.json

high-saliency occlusion이 low/random control보다 일관되게 강하지 않으면
Grad-CAM 공간 특징은 탐색적 상관 분석으로만 보고 원인으로 해석하지 않습니다.
